In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import json

def get_lora_deltas(repo_name: str, lora_alpha: int = None, r: int = None) -> dict:
    """Compute ΔW = lora_B @ lora_A * (alpha/r) directly from adapter weights.

    BUG FIX: lora_alpha and r are no longer hardcoded (32, 16). They are read
    from the adapter's own adapter_config.json unless explicitly overridden.
    Previously every adapter was assumed to share the same (alpha=32, r=16),
    which silently mis-scales ΔW for any adapter trained with a different
    config — this directly distorts that adapter's magnitude relative to the
    others before any merge method even runs.
    """
    config_path = hf_hub_download(repo_id=f"Srishtik/{repo_name}", filename="adapter_config.json")
    cfg = json.load(open(config_path))

    actual_r     = r if r is not None else cfg.get("r")
    actual_alpha = lora_alpha if lora_alpha is not None else cfg.get("lora_alpha")
    if actual_r is None or actual_alpha is None:
        raise ValueError(f"Could not determine r/lora_alpha for {repo_name} from adapter_config.json: {cfg}")

    path = hf_hub_download(
        repo_id  = f"Srishtik/{repo_name}",
        filename = "adapter_model.safetensors"
    )
    adapter_weights = load_file(path)
    scale = actual_alpha / actual_r

    # Group A and B matrices by layer
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()

    # Compute ΔW for each layer
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])  # (d_out, d_in)

    print(f"  {repo_name:<35} r={actual_r}  alpha={actual_alpha}  scale={scale:.4f}")
    return deltas


# ── Usage ──
# NOTE: variables renamed from the previous reasoning_deltas/non_reasoning_deltas/ag_deltas
# (leftover names from the AG News/OpenMath/FineTome experiment) to reflect what they
# actually hold in this experiment — a real risk for future edit mistakes otherwise.
print("Loading adapters (r/alpha read from each repo's own config):")
dolly_deltas      = get_lora_deltas("qwen3-trained-on-dolly-15k")
metamath_deltas   = get_lora_deltas("qwen3-trained-on-metamath-15k")
codealpaca_deltas = get_lora_deltas("qwen3-trained-on-code-alpaca-18k")

print(f"\nAdapter 1 (dolly) layers    : {len(dolly_deltas)}")
print(f"Adapter 2 (metamath) layers : {len(metamath_deltas)}")
print(f"Adapter 3 (codealpaca) layers: {len(codealpaca_deltas)}")
print(f"Sample keys: {list(dolly_deltas.keys())[:3]}")


## Normalize model keys before merging

There is a mismatch between the keys of base model and the trained model. So we have to normalize them and then merge them or else it will show 0

In [ ]:
from transformers import AutoModelForCausalLM
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen3-0.6B",
    torch_dtype=torch.float16,
    device_map="cpu",
)
base_sd = {k: v.cpu() for k, v in base_model.state_dict().items()}
del base_model
torch.cuda.empty_cache()

print(f"Base keys: {len(base_sd)}")
print(f"Sample base keys: {list(base_sd.keys())[:3]}")

In [ ]:
def normalize_delta_keys(deltas: dict) -> dict:
    """
    Convert 'base_model.model.model.layers.0.self_attn.q_proj.'
    to 'model.layers.0.self_attn.q_proj.weight'
    """
    normalized = {}
    for k, v in deltas.items():
        new_key = k.replace("base_model.model.", "")  # strip LoRA prefix
        new_key = new_key.rstrip(".")                  # remove trailing dot
        new_key = new_key + ".weight"                  # add back .weight suffix
        normalized[new_key] = v
    return normalized

dolly_deltas      = normalize_delta_keys(dolly_deltas)
metamath_deltas   = normalize_delta_keys(metamath_deltas)
codealpaca_deltas = normalize_delta_keys(codealpaca_deltas)

# Verify
print(f"Sample normalized key: {list(dolly_deltas.keys())[:3]}")
print(f"Overlap with base_sd: {len(set(dolly_deltas.keys()) & set(base_sd.keys()))}")  # expect 196 (28 layers × 7 target modules for Qwen3-0.6B)


In [ ]:
deltas = []
deltas.append(dolly_deltas)
deltas.append(metamath_deltas)
deltas.append(codealpaca_deltas)

adapter_names = ["dolly", "metamath", "codealpaca"]  # index-aligned with `deltas` — used for SLERP order-naming below


In [ ]:
len(deltas)

In [ ]:


from copy import deepcopy
import torch.nn.functional as F
from typing import Dict, Optional

def apply_delta_to_base(base_state_dict,merged_deltas):
    merged=deepcopy(base_state_dict)
    for key in merged_deltas:
        if key in merged:
            merged[key]=(base_state_dict[key].float()+merged_deltas[key]).to(base_state_dict[key].dtype)
    return merged



## Linear Merge

In [ ]:
def linear_merge(deltas: list[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    n = len(deltas)
    weights = [1.0 / n] * n

    all_keys = set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys &= set(d.keys())

    merged = {}
    for key in all_keys:
        merged[key] = torch.zeros_like(deltas[0][key].float())
        for weight, delta in zip(weights, deltas):
            merged[key] += weight * delta[key].float()

    return merged

## SVD Merge

In [ ]:
def svd_merge(
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    rank: Optional[int] = None,
) -> Dict[str, torch.Tensor]:
    """
    Linearly combine deltas, then SVD-truncate the result to `rank`.
    """
    n = len(deltas)
    if weights is None:
        weights = [1.0 / n] * n

    keys = set(deltas[0].keys())
    for d in deltas[1:]:
        keys &= set(d.keys())

    merged = {}
    for key in keys:
        combined = torch.zeros_like(deltas[0][key].float())
        for weight, delta in zip(weights, deltas):
            combined += weight * delta[key].float()

        if combined.dim() < 2:
            merged[key] = combined
            continue

        try:
            U, S, Vh = torch.linalg.svd(combined, full_matrices=False)
            r = rank if rank is not None else S.shape[0]
            r = min(r, S.shape[0])
            merged[key] = (U[:, :r] * S[:r].unsqueeze(0)) @ Vh[:r, :]
        except Exception:
            merged[key] = combined

    return merged

## TIES MERGE

In [ ]:
def ties_merge(deltas:list[Dict[str,torch.Tensor]],weights:Optional[list[float]]=None,density:float=0.2)->Dict[str,torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())
    merged={}
    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        
        ## TRIM
        trimmed=[]
        for t in tensors:
            flat=t.abs().flatten()
            if flat.numel()==0:
                trimmed.append(t)
                continue
            k=max(1,int(density*flat.numel()))
            threshold=torch.topk(flat,k).values.min()
            mask=t.abs()>=threshold
            trimmed.append(t*mask)
            
    ## ELECT 
        sign_sum=sum(torch.sign(t) for t in trimmed)
        elected_sign=torch.sign(sign_sum)
        elected_sign[elected_sign==0]=1.0
    
    ## MERGE
        numerator=torch.zeros_like(tensors[0]) # Sum of accepted weighted updates
        denominator=torch.zeros_like(tensors[0]) # Total weight of accepted adapters

        for w, t in zip(weights,trimmed):
            agree_mask=(torch.sign(t)==elected_sign).float()
            numerator+=w*t*agree_mask
            denominator+=w*agree_mask
        denominator=torch.clamp(denominator,min=1e-6)
        merged[key]=numerator/denominator
    
    return merged

## DARE MERGE

In [ ]:
def dare_merge(
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    density: float = 0.2,
    seed: int = 42,
) -> Dict[str, torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())

    merged={}
    rng=torch.Generator()
    rng.manual_seed(seed)

    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        result=torch.zeros_like(tensors[0])

        for w,t in zip(weights,tensors):
            mask=torch.bernoulli(
                torch.full(t.shape,density),generator=rng
            ).to(t.device)
            dare_delta=t*mask/(density+1e-8)
            result+=w*dare_delta
        merged[key]=result
    return merged

 ## SLERP Merge

In [ ]:
from typing import Union, Optional, Dict
import torch

def slerp_merge(
    deltas: list,
    t: Optional[Union[float, list]] = None,
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    """
    SLERP merge — direction blend unchanged (confirmed correct: SLERP's cosine
    similarity to each original adapter was consistently HIGHER than linear's,
    everywhere). Only the norm interpolation is fixed.
 
    v1 (my first attempt) was WRONG: it multiplied the sin-based coefficients
    (which already encode the t-weighting via sin((1-t)*omega) / sin(t*omega))
    by an additional (1-t)/t factor — double-counting the weight and collapsing
    the norm to 39.8% of metamath's original magnitude (worse than the original
    bug's 112.5% inflation, just in the other direction).
 
    v2 (this fix): keep the direction blend exactly as before (c1*v1u + c2*v2u,
    which has norm 1 by construction), and blend the two ORIGINAL magnitudes
    using a geometric mean instead of a linear (arithmetic) mean:
        interp_norm = n1^(1-t) * n2^t
    This still satisfies the correct boundary conditions (t=0 -> n1, t=1 -> n2),
    but by AM-GM is always <= the linear blend used previously -- a principled
    way to be more conservative for near-orthogonal vectors, rather than an
    ad hoc rescale.
    """
    n = len(deltas)
    assert n >= 2
    if isinstance(t, float):
        assert n == 2
        t = [t]
    if t is None:
        t = [1.0 / (i + 2) for i in range(n - 1)]
    assert len(t) == n - 1
 
    def slerp_pair(d1, d2, t_val):
        merged = {}
        keys = set(d1.keys()) & set(d2.keys())
        for key in keys:
            v1 = d1[key].float().flatten()
            v2 = d2[key].float().flatten()
            original_shape = d1[key].shape
            n1, n2 = torch.norm(v1), torch.norm(v2)
            if n1 < eps or n2 < eps:
                merged[key] = (1 - t_val) * d1[key].float() + t_val * d2[key].float()
                continue
            v1u, v2u = v1 / n1, v2 / n2
            dot = torch.clamp(torch.dot(v1u, v2u), -1.0 + eps, 1.0 - eps)
            omega = torch.acos(dot)
            sin_omega = torch.sin(omega)
            if sin_omega.abs() < eps:
                merged[key] = (1 - t_val) * d1[key].float() + t_val * d2[key].float()
            else:
                c1 = torch.sin((1 - t_val) * omega) / sin_omega
                c2 = torch.sin(t_val * omega) / sin_omega
                direction = c1 * v1u + c2 * v2u  # norm ≈ 1, unchanged from original
                # FIX: geometric-mean norm blend instead of linear blend
                interp_norm = (n1 ** (1 - t_val)) * (n2 ** t_val)
                merged[key] = (direction * interp_norm).reshape(original_shape)
        return merged
 
    result = deltas[0]
    for i in range(1, n):
        result = slerp_pair(result, deltas[i], t[i - 1])
    return result
 



## BWSum Merge (Balanced Weighted Subspace Union Merging)

In [ ]:
from typing import Union, Optional, Dict
import torch


def balanced_weighted_subspace_union_merge(
    deltas: list[Dict[str, torch.Tensor]],
    t: Optional[Union[float, list[float]]] = None,
    rank: Optional[int] = 16,
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    n = len(deltas)
    assert n >= 2, "Weighted subspace union merge expects at least 2 deltas"

    if isinstance(t, float):
        t = [t]
    if t is None:
        t = [1.0 / (i + 2) for i in range(n - 1)]
    assert len(t) == n - 1, f"Expected {n-1} t values, got {len(t)}"

    r = rank if rank is not None else 16

    def low_rank_svd(w: torch.Tensor, k: int):
        """Randomized SVD — only computes top-k components."""
        U, S, Vh = torch.svd_lowrank(w, q=k, niter=4)
        return U, S, Vh.T  # svd_lowrank returns V not Vh

    def merge_pair(
        d1: Dict[str, torch.Tensor],
        d2: Dict[str, torch.Tensor],
        t_val: float,          # weight for d2; d1 gets (1 - t_val)
    ) -> Dict[str, torch.Tensor]:
        merged = {}
        keys = set(d1.keys()) & set(d2.keys())

        for key in keys:
            w1 = d1[key].float()
            w2 = d2[key].float()
            original_shape = w1.shape

            # Non-matrix tensors: plain weighted average
            if w1.dim() != 2:
                merged[key] = ((1 - t_val) * w1 + t_val * w2).reshape(original_shape)
                continue

            rank_eff = min(r, w1.shape[0], w1.shape[1])

            # ----------------------------------------------------------------
            # Step 1: SVD each adapter
            # ----------------------------------------------------------------
            U1, S1, Vh1 = low_rank_svd(w1, rank_eff)  # (d_out,r),(r,),(r,d_in)
            U2, S2, Vh2 = low_rank_svd(w2, rank_eff)

            # ----------------------------------------------------------------
            # Step 2+3: Energy embedding + weighting
            # w1 gets weight (1-t), w2 gets weight t
            # Distribute sqrt(weight) into both A and B so A@B^T = weight * W
            # ----------------------------------------------------------------
            w1_scale = (1.0 - t_val) ** 0.5
            w2_scale = t_val ** 0.5

            A1 = w1_scale * U1 * S1.sqrt()          # (d_out, r)
            B1 = w1_scale * Vh1.T * S1.sqrt()       # (d_in,  r)  — V, not Vh
            A2 = w2_scale * U2 * S2.sqrt()          # (d_out, r)
            B2 = w2_scale * Vh2.T * S2.sqrt()       # (d_in,  r)

            # ----------------------------------------------------------------
            # Step 4: Reserve rank budget proportional to t
            # r1 columns reserved for adapter 1, r2 for adapter 2
            # ----------------------------------------------------------------
            r1 = max(1, round((1.0 - t_val) * rank_eff))
            r2 = rank_eff - r1  # guarantees r1 + r2 == rank_eff

            # ----------------------------------------------------------------
            # Step 5: Keep top-r1 / top-r2 directions (already sorted by SVD)
            # ----------------------------------------------------------------
            A1_r1 = A1[:, :r1]   # (d_out, r1)
            B1_r1 = B1[:, :r1]   # (d_in,  r1)
            A2_r2 = A2[:, :r2]   # (d_out, r2)
            B2_r2 = B2[:, :r2]   # (d_in,  r2)

            # ----------------------------------------------------------------
            # Step 6: Form union  [A1|A2], [B1|B2]  — shapes (d_out, r), (d_in, r)
            # ----------------------------------------------------------------
            A_union = torch.cat([A1_r1, A2_r2], dim=1)   # (d_out, rank_eff)
            B_union = torch.cat([B1_r1, B2_r2], dim=1)   # (d_in,  rank_eff)

            # ----------------------------------------------------------------
            # Step 7: Orthogonalize via QR
            # ----------------------------------------------------------------
            Q_A, R_A = torch.linalg.qr(A_union)   # Q_A:(d_out,rank_eff)
            Q_B, R_B = torch.linalg.qr(B_union)   # Q_B:(d_in, rank_eff)

            # ----------------------------------------------------------------
            # Step 8: Small SVD in union space  M = R_A @ R_B^T
            # ----------------------------------------------------------------
            M = R_A @ R_B.T                        # (rank_eff, rank_eff)
            # Full SVD on the tiny (rank_eff × rank_eff) matrix — cheap
            U_tilde, Sigma, Vh_tilde = torch.linalg.svd(M, full_matrices=False)

            # ----------------------------------------------------------------
            # Step 9: Reconstruct merged weight matrix
            # ----------------------------------------------------------------
            U_merge  = Q_A @ U_tilde               # (d_out, rank_eff)
            V_merge  = Q_B @ Vh_tilde.T            # (d_in,  rank_eff)
            W_merge  = (U_merge * Sigma) @ V_merge.T  # (d_out, d_in)

            merged[key] = W_merge.reshape(original_shape)

        return merged

    # Sequential pairwise merge with streaming-mean t schedule
    result = deltas[0]
    for i in range(1, n):
        result = merge_pair(result, deltas[i], t[i - 1])

    return result

In [ ]:
def merge_adapters(
    method: str,
    base_state_dict: Dict[str, torch.Tensor],
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    **kwargs,
) -> Dict[str, torch.Tensor]:
    """
    Unified entry point for all merge methods.

    Args:
        method:                one of ['linear', 'svd', 'ties', 'dare', 'slerp', 'bwsum']
        base_state_dict:       base model weights
        deltas:                list of per-adapter ΔW state dicts
        weights:               per-model weights (default: uniform). NOTE: ignored by
                                'slerp' and 'bwsum', which derive equal weighting from
                                their own `t` schedule instead — see slerp_merge's docstring.
        **kwargs:              method-specific args (density, rank, t, seed, etc.)

    Returns:
        merged state dict (ready to load into model)
    """
    n = len(deltas)
    if weights is None:
        weights = [1.0 / n] * n

    if method == "linear":
        merged_delta = linear_merge(deltas)

    elif method == "svd":
        merged_delta = svd_merge(
            deltas,
            rank=kwargs.get("rank", None)
        )

    elif method == "ties":
        merged_delta = ties_merge(deltas, weights=weights, density=kwargs.get("density", 0.2))

    elif method == "dare":
        merged_delta = dare_merge(
            deltas, weights=weights,
            density=kwargs.get("density", 0.2),
            seed=kwargs.get("seed", 42)
        )

    elif method == "dare_ties":
        # BUG FIX: this branch previously called an undefined dare_ties_merge()
        # (NameError if ever invoked). Not currently used by any `methods` list,
        # but fails loudly and clearly now instead of silently existing as dead code.
        raise NotImplementedError(
            "dare_ties_merge is not implemented. Remove 'dare_ties' from your methods "
            "list, or implement dare_ties_merge() before calling this."
        )

    elif method == "slerp":
        n = len(deltas)
        if n == 2:
            t_default = 0.5
        else:
            t_default = [1.0 / (i + 2) for i in range(n - 1)]
        merged_delta = slerp_merge(deltas, t=kwargs.get("t", t_default))

    elif method == "bwsum":
        merged_delta = balanced_weighted_subspace_union_merge(
            deltas,
            t=kwargs.get("t", None),
            rank=kwargs.get("rank", 16),
        )

    else:
        raise ValueError(f"Unknown method: {method}. Choose from: linear, svd, ties, dare, slerp, bwsum")

    return apply_delta_to_base(base_state_dict, merged_delta)


In [ ]:
def upload_merged_model(
    merged_sd: dict,
    method: str,
    tokenizer,
    hf_token: str,
    base_repo: str = "unsloth/Qwen3-0.6B",
    your_hf_username: str = "Srishtik",
    max_seq_length: int = 2048,
    dtype=torch.float16,
    push_to_hub: bool = True,
    save_local: bool = False,
    local_dir: str = "./merged_models",
    repo_name_override: Optional[str] = None,
):
    """
    Loads merged state dict into a fresh base model and uploads to HuggingFace.

    Args:
        merged_sd           : output of merge_adapters()
        method               : merge method name — used for repo naming
        tokenizer            : tokenizer from your training run
        hf_token             : your HuggingFace write token
        base_repo            : base model to load architecture from
        your_hf_username     : your HF username
        max_seq_length       : must match training config
        dtype                : float16 recommended for upload
        push_to_hub          : whether to push to HF Hub
        save_local           : whether to also save locally
        local_dir            : parent dir for local saves
        repo_name_override   : NEW — if set, used verbatim as the repo name instead of
                               the default f"{username}/Qwen3-0.6B-{method}-3-adapters-merged-2".
                               Needed for the SLERP order-sweep, where 6 permutations of the
                               same method would otherwise collide on one repo name.
    """
    import os
    from unsloth import FastLanguageModel

    repo_name = repo_name_override if repo_name_override is not None else \
        f"{your_hf_username}/Qwen3-0.6B-{method}-3-adapters-merged-2"
    print(f"[upload] Preparing model for method='{method}' → {repo_name}")

    # ── Load fresh base model to receive merged weights ──
    model, _ = FastLanguageModel.from_pretrained(
        model_name     = base_repo,
        max_seq_length = max_seq_length,
        load_in_4bit   = False,
        dtype          = dtype,
    )

    # ── Cast merged_sd to match model dtype before loading ──
    target_dtype = next(model.parameters()).dtype
    cast_sd = {
        k: v.to(target_dtype) if v.is_floating_point() else v
        for k, v in merged_sd.items()
    }

    # ── Load merged weights ──
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys  : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")

    model.eval()

    # ── Save locally ──
    if save_local:
        save_path = os.path.join(local_dir, repo_name.split("/")[-1])
        os.makedirs(save_path, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  [local] Saved to {save_path}")

    # ── Push to HuggingFace Hub ──
    if push_to_hub:
        model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")

    # ── Free memory ──
    del model, cast_sd
    torch.cuda.empty_cache()

    return repo_name


In [ ]:
from unsloth import FastLanguageModel
_,tokenizer=FastLanguageModel.from_pretrained(
    model_name= "unsloth/Qwen3-0.6B",
    max_seq_length=2048,
    load_in_4bit=False,
)

In [ ]:
# HF_TOKEN =key ## Insert your own token
# methods  = ["linear", "svd", "ties", "dare"]

# for method in methods:
    
#     merged_sd = merge_adapters(
#         method = method,
#         base_state_dict = base_sd,
#         deltas = deltas,
#         weights = [1/3,1/3,1/3],
#         density  = 0.2,   # ties / dare 
#         rank = 16,    # svd
#         t  = 0.5,   # slerp
#         seed = 42,    # dare
#     )

#     upload_merged_model(
#         merged_sd = merged_sd,
#         method    = method,
#         tokenizer = tokenizer,
#         hf_token  = HF_TOKEN,
#     )


## SLERP — All Orderings

SLERP is not associative for 3 adapters (see `slerp_merge` docstring above). This sweeps all 3! = 6 permutations of {dolly, metamath, codealpaca} and uploads each as a separately-named model, instead of assuming one arbitrary order is representative.

In [ ]:
import itertools

# ─────────────────────────────────────────────────────────────────────────────
# SLERP ORDER SWEEP
# SLERP is not associative for N=3 (see slerp_merge docstring) — the sequential
# pairwise merge result depends on which two adapters get merged first. Rather
# than pick one arbitrary order and assume it's representative, this runs ALL
# 3! = 6 permutations of {dolly, metamath, codealpaca} and uploads each as a
# separately-named model, so the order effect can be measured empirically
# instead of assumed away.
#
# Naming: Qwen3-0.6B-slerp-order-<first>-<second>-<third>-3-adapters-merged-2
# "first" = the adapter merged first in the pairwise chain (gets compromised
# with "second" before "third" is ever introduced).
# ─────────────────────────────────────────────────────────────────────────────

HF_TOKEN = key  # Insert your own token

delta_by_name = dict(zip(adapter_names, deltas))  # {"dolly": dolly_deltas, "metamath": ..., "codealpaca": ...}

all_slerp_order_repos = []

for perm in itertools.permutations(adapter_names):
    ordered_names  = list(perm)
    ordered_deltas = [delta_by_name[name] for name in ordered_names]

    n = len(ordered_deltas)
    t_value = 0.5 if n == 2 else [1.0 / (i + 2) for i in range(n - 1)]

    print(f"\n{'═'*60}")
    print(f"SLERP order: {' → '.join(ordered_names)}")
    print(f"{'═'*60}")

    merged_sd = merge_adapters(
        method          = "slerp",
        base_state_dict = base_sd,
        deltas          = ordered_deltas,
        weights         = [1/3, 1/3, 1/3],   # ignored by slerp branch, kept for API consistency
        t               = t_value,
    )

    order_suffix = "-".join(ordered_names)
    repo_name = f"Srishtik/Qwen3-0.6B-slerp-order-{order_suffix}-3-adapters-merged-2"

    uploaded_repo = upload_merged_model(
        merged_sd          = merged_sd,
        method              = "slerp",
        tokenizer           = tokenizer,
        hf_token            = HF_TOKEN,
        repo_name_override  = repo_name,
    )

    all_slerp_order_repos.append((ordered_names, uploaded_repo))

print(f"\n{'═'*60}")
print("All 6 SLERP orderings uploaded:")
for ordered_names, repo in all_slerp_order_repos:
    print(f"  {' → '.join(ordered_names):<35} → {repo}")


In [ ]:
# HF_TOKEN =key
# methods = ["bwsum"]
# n = len(deltas)
# t_value = 0.5 if n == 2 else [1.0 / (i + 2) for i in range(n - 1)]

# for method in methods:
#     merged_sd = merge_adapters(
#         method          = method,
#         base_state_dict = base_sd,
#         deltas          = deltas,
#         weights         = [1/3, 1/3, 1/3],
#         density         = 0.2,
#         t               = t_value,   # ← now correctly [0.5, 0.5] for 3 deltas
#         seed            = 42,
#     )
#     upload_merged_model(
#         merged_sd = merged_sd,
#         method    = method,
#         tokenizer = tokenizer,
#         hf_token  = HF_TOKEN,
#     )
